In [0]:
# Buscar tablas de la capa Gold en Unity Catalog
# Identifica esquemas cuyos nombres contengan 'gold' y lista sus tablas

from pyspark.sql import functions as F

# Listar todos los catálogos disponibles
catalogs = spark.sql("SHOW CATALOGS").collect()
print("Catálogos disponibles:")
for c in catalogs:
    print(f"  - {c['catalog']}")

print("\n" + "="*80 + "\n")

# Recorrer catálogos y esquemas buscando los de la capa Gold
for c in catalogs:
    catalog_name = c['catalog']
    try:
        schemas = spark.sql(f"SHOW SCHEMAS IN {catalog_name}").collect()
        for s in schemas:
            schema_name = s['databaseName']
            if 'gold' in schema_name.lower():
                print(f"Catálogo: {catalog_name} | Esquema Gold: {schema_name}")
                tables = spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}").collect()
                for t in tables:
                    print(f"  - {catalog_name}.{schema_name}.{t['tableName']}")
                print()
    except Exception as e:
        # Saltar catálogos a los que no se tiene acceso
        pass

In [0]:
# ===========================================================================
# Dashboard de Calidad de Datos – Capa Gold
# ===========================================================================
# Métricas críticas evaluadas:
#   1. Completitud (% de valores nulos por columna)
#   2. Duplicidad (registros duplicados por tabla)
#   3. Volumetría (cantidad de filas por tabla)
#   4. Unicidad (duplicados en columnas llave)
#   5. Consistencia de tipos (validación de esquema)
# ===========================================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from pyspark.sql import functions as F

# -------------------------------------------------------------------------
# 1. Descubrir tablas Gold (reutiliza la lógica de la celda anterior)
# -------------------------------------------------------------------------
gold_tables = []
catalogs = spark.sql("SHOW CATALOGS").collect()

for c in catalogs:
    catalog_name = c['catalog']
    try:
        schemas = spark.sql(f"SHOW SCHEMAS IN {catalog_name}").collect()
        for s in schemas:
            schema_name = s['databaseName']
            if 'gold' in schema_name.lower():
                tables = spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}").collect()
                for t in tables:
                    if not t['isTemporary']:
                        gold_tables.append(f"{catalog_name}.{schema_name}.{t['tableName']}")
    except Exception:
        pass

print(f"Tablas Gold encontradas: {len(gold_tables)}")
for t in gold_tables:
    print(f"  - {t}")

# -------------------------------------------------------------------------
# 2. Calcular métricas de calidad por tabla
# -------------------------------------------------------------------------
quality_results = []  # lista de dicts: {tabla, fila, columna, nulos, pct_nulos}
table_summary = []    # resumen por tabla

def es_columna_llave(nombre_col):
    """Heurística para identificar columnas llave."""
    n = nombre_col.lower()
    return (
        n.endswith("_id") or n.endswith("_key") or n.endswith("_pk")
        or n == "id" or n.startswith("id_") or n.endswith("id") and len(n) <= 10
    )

for full_name in gold_tables:
    try:
        df = spark.table(full_name)
        total_rows = df.count()
        columns = df.columns
        schema = df.schema

        # --- 2a. Completitud (nulos por columna) ---
        null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in columns]
        null_counts_row = df.agg(*null_exprs).collect()[0]

        for c in columns:
            nulls = null_counts_row[c]
            pct = (nulls / total_rows * 100) if total_rows > 0 else 0.0
            quality_results.append({
                "tabla": full_name,
                "columna": c,
                "nulos": nulls,
                "pct_nulos": round(pct, 2),
                "pct_completitud": round(100 - pct, 2),
                "tipo_dato": str(schema[c].dataType),
                "es_llave": es_columna_llave(c),
            })

        # --- 2b. Duplicidad global (todas las columnas) ---
        dup_count = total_rows - df.dropDuplicates().count()

        # --- 2c. Unicidad en columnas llave candidatas ---
        key_cols = [c for c in columns if es_columna_llave(c)]
        key_dups = 0
        if key_cols:
            key_dups = total_rows - df.dropDuplicates(key_cols).count()

        # --- 2d. Resumen de la tabla ---
        cols_criticas = [
            r for r in quality_results if r["tabla"] == full_name and r["pct_nulos"] > 0
        ]
        table_summary.append({
            "tabla": full_name.split(".")[-1],
            "full_name": full_name,
            "filas": total_rows,
            "columnas": len(columns),
            "duplicados": dup_count,
            "pct_duplicados": round(dup_count / total_rows * 100, 2) if total_rows else 0,
            "dup_llaves": key_dups,
            "pct_dup_llaves": round(key_dups / total_rows * 100, 2) if total_rows else 0,
            "cols_con_nulos": len(cols_criticas),
        })

    except Exception as e:
        print(f"⚠️  Error evaluando {full_name}: {e}")

# -------------------------------------------------------------------------
# 3. DataFrames de resultados
# -------------------------------------------------------------------------
quality_df = pd.DataFrame(quality_results)
summary_df = pd.DataFrame(table_summary)

# Métricas globales
total_tablas = len(gold_tables)
total_filas = int(summary_df["filas"].sum()) if not summary_df.empty else 0
total_duplicados = int(summary_df["duplicados"].sum()) if not summary_df.empty else 0
total_dup_llaves = int(summary_df["dup_llaves"].sum()) if not summary_df.empty else 0
promedio_completitud = (
    round(quality_df["pct_completitud"].mean(), 2) if not quality_df.empty else 100.0
)

display(summary_df)

# -------------------------------------------------------------------------
# 4. Dashboard visual (5 paneles)
# -------------------------------------------------------------------------
plt.style.use("seaborn-v0_8-darkgrid")
fig = plt.figure(figsize=(20, 22))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3)

fig.suptitle("📊 Dashboard de Calidad de Datos – Capa Gold",
             fontsize=22, fontweight="bold", y=0.98)

# ---- Panel 1: KPIs globales (tarjetas de texto) ----
ax_kpi = fig.add_subplot(gs[0, :])
ax_kpi.axis("off")
kpis = [
    ("Tablas evaluadas", f"{total_tablas}", "#2980B9"),
    ("Total de filas", f"{total_filas:,}", "#27AE60"),
    ("Promedio completitud", f"{promedio_completitud}%", "#8E44AD"),
    ("Duplicados totales", f"{total_duplicados:,}", "#E74C3C"),
    ("Dup. en llaves", f"{total_dup_llaves:,}", "#F39C12"),
]
x_start = 0.03
card_w = 0.17
for i, (label, value, color) in enumerate(kpis):
    x = x_start + i * (card_w + 0.035)
    ax_kpi.add_patch(plt.Rectangle((x, 0.35), card_w, 0.55,
                                    transform=ax_kpi.transAxes,
                                    facecolor=color, alpha=0.12,
                                    edgecolor=color, linewidth=2))
    ax_kpi.text(x + card_w / 2, 0.72, value, transform=ax_kpi.transAxes,
                fontsize=26, fontweight="bold", ha="center", va="center", color=color)
    ax_kpi.text(x + card_w / 2, 0.45, label, transform=ax_kpi.transAxes,
                fontsize=12, ha="center", va="center", color="#555555")

# ---- Panel 2: Volumetría por tabla ----
ax2 = fig.add_subplot(gs[1, 0])
if not summary_df.empty:
    bars = ax2.barh(summary_df["tabla"], summary_df["filas"], color="#2980B9", edgecolor="white")
    ax2.set_xlabel("Cantidad de filas")
    ax2.set_title("Volumetría por tabla", fontsize=14, fontweight="bold")
    ax2.invert_yaxis()
    ax2.tick_params(axis="y", labelsize=9)
    for bar in bars:
        w = bar.get_width()
        ax2.text(w + max(summary_df["filas"]) * 0.01, bar.get_y() + bar.get_height() / 2,
                 f"{int(w):,}", va="center", fontsize=8, color="#333")
else:
    ax2.text(0.5, 0.5, "Sin datos", ha="center", va="center", transform=ax2.transAxes)

# ---- Panel 3: % Duplicados por tabla ----
ax3 = fig.add_subplot(gs[1, 1])
if not summary_df.empty:
    colors_d = ["#E74C3C" if d > 0 else "#27AE60" for d in summary_df["pct_duplicados"]]
    ax3.barh(summary_df["tabla"], summary_df["pct_duplicados"], color=colors_d, edgecolor="white")
    ax3.set_xlabel("% Duplicados")
    ax3.set_title("Duplicidad por tabla (%)", fontsize=14, fontweight="bold")
    ax3.invert_yaxis()
    ax3.tick_params(axis="y", labelsize=9)
else:
    ax3.text(0.5, 0.5, "Sin datos", ha="center", va="center", transform=ax3.transAxes)

# ---- Panel 4: Top 15 columnas con mayor % de nulos ----
ax4 = fig.add_subplot(gs[2, 0])
if not quality_df.empty:
    top_nulls = quality_df.nlargest(15, "pct_nulos")
    top_nulls = top_nulls[top_nulls["pct_nulos"] > 0]
    if not top_nulls.empty:
        labels = [f"{row['tabla'].split('.')[-1]}.{row['columna']}" for _, row in top_nulls.iterrows()]
        colors_n = ["#E74C3C" if r >= 50 else "#F39C12" if r >= 10 else "#F1C40F"
                    for r in top_nulls["pct_nulos"]]
        ax4.barh(labels, top_nulls["pct_nulos"], color=colors_n, edgecolor="white")
        ax4.set_xlabel("% Nulos")
        ax4.set_title("Top 15 columnas con mayor % de nulos", fontsize=14, fontweight="bold")
        ax4.invert_yaxis()
        ax4.tick_params(axis="y", labelsize=8)
    else:
        ax4.text(0.5, 0.5, "✅ Sin nulos significativos", ha="center", va="center",
                transform=ax4.transAxes, fontsize=14, color="green")
else:
    ax4.text(0.5, 0.5, "Sin datos", ha="center", va="center", transform=ax4.transAxes)

# ---- Panel 5: Completitud promedio por tabla (heatmap de barras) ----
ax5 = fig.add_subplot(gs[2, 1])
if not quality_df.empty:
    comp_by_table = (
        quality_df.groupby("tabla")["pct_completitud"]
        .mean()
        .round(2)
        .sort_values()
    )
    short_names = [t.split(".")[-1] for t in comp_by_table.index]
    colors_c = ["#E74C3C" if v < 90 else "#F39C12" if v < 99 else "#27AE60" for v in comp_by_table.values]
    ax5.barh(short_names, comp_by_table.values, color=colors_c, edgecolor="white")
    ax5.set_xlabel("% Completitud promedio")
    ax5.set_title("Completitud promedio por tabla", fontsize=14, fontweight="bold")
    ax5.invert_yaxis()
    ax5.tick_params(axis="y", labelsize=9)
    ax5.set_xlim(0, 105)
    for i, v in enumerate(comp_by_table.values):
        ax5.text(v + 1, i, f"{v}%", va="center", fontsize=8, color="#333")
else:
    ax5.text(0.5, 0.5, "Sin datos", ha="center", va="center", transform=ax5.transAxes)

plt.savefig("/tmp/dashboard_calidad_datos.png", dpi=150, bbox_inches="tight")
plt.show()

# -------------------------------------------------------------------------
# 5. Reporte detallado de columnas críticas (nulos > 0 o duplicados en llave)
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print("REPORTE DETALLADO DE CALIDAD DE DATOS")
print("=" * 80)

if not quality_df.empty:
    criticas = quality_df[quality_df["pct_nulos"] > 0].sort_values("pct_nulos", ascending=False)
    if not criticas.empty:
        print(f"\n🔴 Columnas con valores nulos ({len(criticas)} columnas):")
        for _, row in criticas.iterrows():
            severidad = "CRÍTICO" if row["pct_nulos"] >= 50 else "ALTO" if row["pct_nulos"] >= 10 else "MEDIO"
            print(f"  [{severidad:8s}] {row['tabla'].split('.')[-1]}.{row['columna']} "
                  f"→ {row['nulos']:,} nulos ({row['pct_nulos']}%)")
    else:
        print("\n✅ No se encontraron columnas con valores nulos.")

if not summary_df.empty:
    dup_tabs = summary_df[summary_df["duplicados"] > 0]
    if not dup_tabs.empty:
        print(f"\n🔴 Tablas con registros duplicados:")
        for _, row in dup_tabs.iterrows():
            print(f"  - {row['tabla']}: {row['duplicados']:,} duplicados ({row['pct_duplicados']}%)")
    else:
        print("\n✅ No se encontraron registros duplicados.")

    dup_key_tabs = summary_df[summary_df["dup_llaves"] > 0]
    if not dup_key_tabs.empty:
        print(f"\n🔴 Duplicidad en columnas llave:")
        for _, row in dup_key_tabs.iterrows():
            print(f"  - {row['tabla']}: {row['dup_llaves']:,} duplicados en llave ({row['pct_dup_llaves']}%)")
    else:
        print("\n✅ No se encontraron duplicados en columnas llave.")

print("\n" + "=" * 80)
print("Dashboard generado correctamente. Revisa los gráficos arriba.")
print("=" * 80)